# 2. Positive definite matrices  [2.6–2.7, 3.1, 3.5]

TOML-MIRI — Linear Algebra for Data Science, exploration exercises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

np.set_printoptions(precision=4, suppress=True)

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

---
## 2.1 The shape of a quadratic form

**Task.** Plot the surface and contour lines of $f(x) = x^\top S x$ with $S = \begin{bmatrix} a & b\\ b & c\end{bmatrix}$, with sliders for $a, b, c$. Show the eigenvalues and draw the eigenvectors on the contour plot.

### Prediction (write this *before* running)

- For $S = I$ the surface is ___ and the contours are ___.
- For $S = \mathrm{diag}(1, 0)$ the surface is ___.
- What sign pattern of eigenvalues gives a saddle?

*(fill in)*

In [ ]:
def classify(lam, tol=1e-10):
    if np.all(lam > tol):
        return "positive definite (bowl)"
    if np.all(lam < -tol):
        return "negative definite (upside-down bowl)"
    if np.all(lam >= -tol):
        return "positive semi-definite (valley with a flat line)"
    if np.all(lam <= tol):
        return "negative semi-definite (ridge with a flat line)"
    return "indefinite (saddle)"

g = np.linspace(-2, 2, 101)
X1, X2 = np.meshgrid(g, g)

def quad_form(S, X1, X2):
    return S[0, 0] * X1**2 + 2 * S[0, 1] * X1 * X2 + S[1, 1] * X2**2

def plot_quadratic_form(a, b, c):
    S = np.array([[a, b], [b, c]])
    lam, Q = np.linalg.eigh(S)                 # ascending eigenvalues, orthonormal Q
    F = quad_form(S, X1, X2)

    fig = plt.figure(figsize=(13, 5.5))
    ax3 = fig.add_subplot(1, 2, 1, projection="3d")
    ax3.plot_surface(X1, X2, F, cmap="viridis", alpha=0.9, linewidth=0)
    ax3.set_xlabel("x₁"); ax3.set_ylabel("x₂"); ax3.set_zlabel("f(x)")
    ax3.set_title(f"S = [[{a:g}, {b:g}], [{b:g}, {c:g}]]")

    ax = fig.add_subplot(1, 2, 2)
    if np.ptp(F) > 1e-12:
        cs = ax.contour(X1, X2, F, levels=15, cmap="viridis")
        ax.clabel(cs, fontsize=7)
    for i, col in enumerate(["crimson", "royalblue"]):
        q = Q[:, i]
        ax.annotate("", xy=1.5 * q, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", color=col, lw=2.5))
        ax.text(*(1.75 * q), f"q{i+1}\nλ={lam[i]:.2f}", color=col, ha="center", va="center")
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_title(f"λ = {lam.round(3)}  →  {classify(lam)}")
    plt.show()

In [ ]:
@interact(a=FloatSlider(min=-3, max=3, step=0.1, value=2.0),
          b=FloatSlider(min=-3, max=3, step=0.1, value=0.5),
          c=FloatSlider(min=-3, max=3, step=0.1, value=1.0))
def explore_quadratic_form(a, b, c):
    plot_quadratic_form(a, b, c)

### Verify numerically

The key identity: with $S = Q\Lambda Q^\top$ and $y = Q^\top x$ (coordinates of $x$ in the eigenvector basis),

$$f(x) = x^\top Q\Lambda Q^\top x = y^\top \Lambda y = \sum_i \lambda_i\, y_i^2 .$$

In [ ]:
S = np.array([[2.0, 0.5], [0.5, 1.0]])
lam, Q = np.linalg.eigh(S)
x = np.random.default_rng(0).standard_normal(2)

check("S = Q Λ Qᵀ", np.allclose(S, Q @ np.diag(lam) @ Q.T))
check("Qᵀ Q = I  (orthonormal eigenvectors)", np.allclose(Q.T @ Q, np.eye(2)))
for i in range(2):
    check(f"f(q{i+1}) = q{i+1}ᵀ S q{i+1} = λ{i+1}", np.isclose(Q[:, i] @ S @ Q[:, i], lam[i]))
check("f(x) = Σ λᵢ (qᵢᵀx)²  for a random x", np.isclose(x @ S @ x, np.sum(lam * (Q.T @ x) ** 2)))

### Explore: find a bowl (PD), a valley with a flat line (PSD), an upside-down bowl (ND) and a saddle (indefinite)

Use the sliders first. Then compare with these settings:

In [ ]:
presets = {"PD": (2, 0.5, 1), "PSD": (1, 1, 1), "ND": (-2, 0.5, -1), "indefinite": (1, 2, 1)}
for name, (a, b, c) in presets.items():
    print(f"--- {name}: a={a}, b={b}, c={c}")
    plot_quadratic_form(a, b, c)

---
### Explain (hand-in — in your own words)

**1. How do the signs of the eigenvalues predict the shape?**
*(Hint: use $f(x) = \sum_i \lambda_i y_i^2$. What does $f$ do along each eigenvector direction?)*

*Your answer.*

**2. Why do the contour ellipses align with the eigenvectors?**
*(Hint: in the coordinates $y = Q^\top x$ the level set $f = c$ reads $\lambda_1 y_1^2 + \lambda_2 y_2^2 = c$. Which axes does that ellipse have, and how long are they?)*

*Your answer.*

---
## 2.2 A sphere becomes an ellipsoid

**Task.** Apply a symmetric PD matrix $S = Q\Lambda Q^\top$ to points $v$ on the unit circle step by step — first $Q^\top$, then $\Lambda$, then $Q$ — and plot the three stages (Figure 5 in the notes).

$Q$ is a rotation by $\theta$, $\Lambda = \mathrm{diag}(\lambda_1, \lambda_2)$. Points are coloured by their angle on the original circle so you can follow each one. Arrows: $e_1, e_2$ (green), $q_1, q_2$ (orange), a sample $v$ (red) — each panel shows their images at that stage.

### Prediction (write this *before* running)

- After $Q^\top$ the circle looks like ___ and $q_1$ is sent to ___.
- The final ellipse has semi-axes of length ___ along directions ___.

*(fill in)*

In [ ]:
def rot(theta_deg):
    t = np.deg2rad(theta_deg)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

phi = np.linspace(0, 2 * np.pi, 120, endpoint=False)
circle = np.vstack([np.cos(phi), np.sin(phi)])

def sphere_to_ellipse(theta=30, lam1=2.0, lam2=0.5):
    Q, Lam = rot(theta), np.diag([lam1, lam2])
    stages = [np.eye(2), Q.T, Lam @ Q.T, Q @ Lam @ Q.T]
    titles = ["v  (unit circle)", "Qᵀv  (rotate)", "ΛQᵀv  (stretch)", "QΛQᵀv = Sv  (rotate back)"]
    v = np.array([np.cos(np.deg2rad(70)), np.sin(np.deg2rad(70))])
    arrows = [(np.array([1.0, 0.0]), "seagreen", "e₁"), (np.array([0.0, 1.0]), "seagreen", "e₂"),
              (Q[:, 0], "darkorange", "q₁"), (Q[:, 1], "darkorange", "q₂"), (v, "crimson", "v")]
    lim = max(1.3, 1.2 * max(lam1, lam2))
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
    for ax, M, title in zip(axes, stages, titles):
        ax.scatter(*(M @ circle), c=phi, cmap="hsv", s=8)
        for vec, col, lab in arrows:
            w = M @ vec
            ax.annotate("", xy=w, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=col, lw=2))
            ax.text(*(w * 1.1 + 0.03), lab, color=col)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
        ax.axhline(0, c="k", lw=0.5); ax.axvline(0, c="k", lw=0.5); ax.grid(alpha=0.3)
        ax.set_title(title)
    fig.suptitle(f"θ = {theta}°,  λ = ({lam1}, {lam2})")
    plt.show()

sphere_to_ellipse()

### Verify numerically

In [ ]:
theta, lam1, lam2 = 30, 2.0, 0.5
Q, Lam = rot(theta), np.diag([lam1, lam2])
S = Q @ Lam @ Q.T
E = S @ circle
radii = np.linalg.norm(E, axis=0)
i_max, i_min = radii.argmax(), radii.argmin()

check("S is symmetric", np.allclose(S, S.T))
check("Q (Λ (Qᵀ v)) = S v  (the three stages compose to S)", np.allclose(Q @ (Lam @ (Q.T @ circle)), E))
check("S qᵢ = λᵢ qᵢ", all(np.allclose(S @ Q[:, i], [lam1, lam2][i] * Q[:, i]) for i in range(2)))
check(f"longest radius of the ellipse = λ_max = {max(lam1, lam2)}", np.isclose(radii.max(), max(lam1, lam2)))
check(f"shortest radius of the ellipse = λ_min = {min(lam1, lam2)}", np.isclose(radii.min(), min(lam1, lam2)))
check("longest axis points along q₁", np.isclose(abs(E[:, i_max] @ Q[:, 0]) / radii[i_max], 1))

### Explore: make one eigenvalue close to 0, then exactly 0

Predict first: what does the ellipse become when $\lambda_2 \to 0$? Is $S$ still PD?

In [ ]:
@interact(theta=FloatSlider(min=0, max=180, step=5, value=30),
          lam1=FloatSlider(min=0, max=3, step=0.05, value=2.0),
          lam2=FloatSlider(min=0, max=3, step=0.05, value=0.5))
def explore_ellipse(theta, lam1, lam2):
    sphere_to_ellipse(theta, lam1, lam2)

---
### Explain (hand-in — in your own words)

**1. What are the semi-axis lengths and directions of the final ellipse?**

*Your answer.*

**2. What happens when one eigenvalue is close to 0, and exactly 0? Relate it to the rank of $S$ and to [2.4].**

*Your answer.*

---
## 2.3 $A^\top A$ is always PSD

**Task.** Generate 1,000 random matrices $A$ (some square, some tall, some rank-deficient) and record the smallest eigenvalue of $A^\top A$ for each. Plot a histogram.

$$x^\top A^\top A x = \|Ax\|_2^2 \ge 0.$$

A fourth family, **wide** matrices (fewer rows than columns), is added for comparison.

### Prediction (write this *before* running)

- Can $\lambda_{\min}(A^\top A)$ ever be negative? What will the computer say?
- Which families will have $\lambda_{\min} = 0$?

*(fill in)*

In [ ]:
rng = np.random.default_rng(0)
kinds = ["square", "tall", "rank-deficient", "wide"]
records = []                                   # (kind, λ_min, A)
for i in range(1000):
    kind = kinds[i % 4]
    n = int(rng.integers(2, 8))
    if kind == "square":
        A = rng.standard_normal((n, n))
    elif kind == "tall":
        A = rng.standard_normal((n + int(rng.integers(1, 6)), n))
    elif kind == "rank-deficient":
        r = int(rng.integers(1, n))                       # rank r < n
        A = rng.standard_normal((n + int(rng.integers(0, 4)), r)) @ rng.standard_normal((r, n))
    else:                                                 # wide: m < n
        A = rng.standard_normal((int(rng.integers(1, n)), n))
    records.append((kind, np.linalg.eigvalsh(A.T @ A).min(), A))

lam_min = np.array([rec[1] for rec in records])
kind_of = np.array([rec[0] for rec in records])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))
ax1.hist([lam_min[kind_of == k] for k in kinds], bins=60, stacked=True, label=kinds)
ax1.set_xlabel("λ_min(AᵀA)"); ax1.set_ylabel("count"); ax1.legend(); ax1.set_title("linear scale")
log_abs = np.log10(np.maximum(np.abs(lam_min), 1e-20))
ax2.hist([log_abs[kind_of == k] for k in kinds], bins=60, stacked=True, label=kinds)
ax2.set_xlabel("log₁₀ |λ_min(AᵀA)|"); ax2.legend(); ax2.set_title("log scale (magnitude)")
plt.show()

for k in kinds:
    v = lam_min[kind_of == k]
    print(f"{k:>15}:  min λ_min = {v.min(): .2e}   max λ_min = {v.max(): .2e}   negatives: {np.sum(v < 0)}")
print(f"\nmost negative λ_min over all 1000 matrices: {lam_min.min():.2e}")

### Connect the zero eigenvalue to the null space

For the eigenvector $v$ of the smallest eigenvalue, $\lambda_{\min} = v^\top A^\top A v = \|Av\|^2$. So $\lambda_{\min} = 0 \iff Av = 0$.

In [ ]:
for kind, lmin, A in records[:8]:
    v = np.linalg.eigh(A.T @ A)[1][:, 0]
    print(f"{kind:>15} {str(A.shape):>7}: λ_min = {lmin: .2e}   ‖A v‖² = {np.linalg.norm(A @ v)**2: .2e}"
          f"   rank {np.linalg.matrix_rank(A)} of n = {A.shape[1]}")

ok = all(np.isclose(np.linalg.norm(A @ np.linalg.eigh(A.T @ A)[1][:, 0]) ** 2, lmin, atol=1e-10)
         for _, lmin, A in records)
check("λ_min = ‖A v_min‖² for all 1000 matrices", ok)
deficient = [np.linalg.matrix_rank(A) < A.shape[1] for _, _, A in records]
check("λ_min ≈ 0 (|λ| < 1e-10) exactly when rank(A) < n",
      all((abs(lmin) < 1e-10) == d for (_, lmin, _), d in zip(records, deficient)))

---
### Explain (hand-in — in your own words)

**1. When is the minimum exactly 0 (up to round-off)?**
*(Hint: compare rank(A) with the number of columns $n$. Why are wide matrices always in that group?)*

*Your answer.*

**2. Connect this to the proof in [2.7] and to the null space of $A$.**
*(Hint: the notes prove PD when $A$ is invertible. What changes in the proof when $Ax = 0$ has a non-zero solution?)*

*Your answer.*

**3. Why can the computed $\lambda_{\min}$ be slightly negative?**

*Your answer.*